In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)

In [3]:
df = pd.read_csv('fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [4]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
X_train

array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       ...,
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ..., 16,  0,  0]], shape=(4800, 784))

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [8]:
X_train

array([[0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.0627451, 0.       ,
        0.       ]], shape=(4800, 784))

In [9]:
# Create custom dataclass
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [10]:
# Create train dataset object
train_dataset = CustomDataset(X_train, y_train)

In [11]:
test_dataset = CustomDataset(X_test, y_test)

In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Dynamic Hyperparam-tuning

In [29]:
# Define NN
class myModel(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, neuron_per_layer, dropout_rate):
        super().__init__()

        layers = []

        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neuron_per_layer))
            layers.append(nn.BatchNorm1d(neuron_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = neuron_per_layer

        layers.append(nn.Linear(neuron_per_layer, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [30]:
# Objective function
def objective(trial):

    # next hyperparams values from the search space
    num_hidden_layers = trial.suggest_int('num_hidden', 1, 5)
    neurons_per_layer = trial.suggest_int('neurons_per_layer', 8, 128, step=8)
    epochs = trial.suggest_int('epochs', 10, 50, step=10)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical('batch_size',  (16, 32, 64, 128))
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # model init
    input_dim = 784
    output_dim = 10

    model = myModel(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate=dropout_rate)

    # optimizer selection
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    elif optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    else:
        pass

    # training loop
    for epoch in range(epochs):
        total_epoch_loss = 0

        for batch_features, batch_label in train_loader:
            # forward
            outputs = model(batch_features)

            # loss
            loss = criterion(outputs, batch_label)

            optimizer.zero_grad()

            # backward
            loss.backward()

            # update grad
            optimizer.step()
        
            total_epoch_loss = total_epoch_loss + loss.item()

        # print(f"Epoch: {epoch+1}, Loss: {total_epoch_loss/(len(train_loader))}")
        
    
    # evaluation
    total = 0
    correct = 0

    with torch.no_grad():
        for batch_features, batch_label in test_loader:
            outputs = model(batch_features)

            _, predicted = torch.max(outputs, 1)

            total += batch_label.shape[0]

            correct += (predicted == batch_label).sum().item()
    accuracy = correct/total
    # print(correct/total)

    return accuracy

In [31]:
import optuna

In [32]:
study = optuna.create_study(direction='maximize')

[I 2026-06-25 11:50:33,644] A new study created in memory with name: no-name-377f600a-f71e-41b4-ae15-9b3d60335a43


In [34]:
study.optimize(objective, n_trials=20)

[I 2026-06-25 11:54:19,955] Trial 10 finished with value: 0.79 and parameters: {'num_hidden': 5, 'neurons_per_layer': 128, 'epochs': 50, 'learning_rate': 0.001982904759530585, 'dropout_rate': 0.1, 'batch_size': 16, 'optimizer': 'SGD', 'weight_decay': 9.916755034886969e-05}. Best is trial 10 with value: 0.79.
[I 2026-06-25 11:55:00,295] Trial 11 finished with value: 0.805 and parameters: {'num_hidden': 5, 'neurons_per_layer': 128, 'epochs': 50, 'learning_rate': 0.0021949461042657962, 'dropout_rate': 0.1, 'batch_size': 16, 'optimizer': 'SGD', 'weight_decay': 0.00011032299092894853}. Best is trial 11 with value: 0.805.
[I 2026-06-25 11:55:41,399] Trial 12 finished with value: 0.7825 and parameters: {'num_hidden': 5, 'neurons_per_layer': 128, 'epochs': 50, 'learning_rate': 0.001470259200737921, 'dropout_rate': 0.1, 'batch_size': 16, 'optimizer': 'SGD', 'weight_decay': 8.735153757345467e-05}. Best is trial 11 with value: 0.805.
[I 2026-06-25 11:56:25,375] Trial 13 finished with value: 0.796

In [35]:
study.best_value

0.815

In [36]:
study.best_params

{'num_hidden': 5,
 'neurons_per_layer': 104,
 'epochs': 50,
 'learning_rate': 1.9854500029696824e-05,
 'dropout_rate': 0.1,
 'batch_size': 32,
 'optimizer': 'RMSprop',
 'weight_decay': 5.472863536153101e-05}

## Static Codes

In [ ]:
# Define NN
class model(nn.Module):
    def __init__(self, feature_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
            # No need softmax. Softmax is by default implemented by cross entropy
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# set learning rate and epochs
epochs = 100
lr = 0.1

In [ ]:
# instatiate the model
model = model(X_train.shape[1])

# loss function
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(model.parameters(), lr, weight_decay=1e-7)

In [ ]:
# Training loop
for epoch in range(epochs):
    total_epoch_loss = 0

    for batch_features, batch_label in train_loader:
        # forward
        outputs = model(batch_features)

        # loss
        loss = criterion(outputs, batch_label)

        optimizer.zero_grad()

        # backward
        loss.backward()

        # update grad
        optimizer.step()
    
        total_epoch_loss = total_epoch_loss + loss.item()

    print(f"Epoch: {epoch+1}, Loss: {total_epoch_loss/(len(train_loader))}")

Epoch: 1, Loss: 1.117295554007093
Epoch: 2, Loss: 0.2625213756412268
Epoch: 3, Loss: 0.21917280004670223
Epoch: 4, Loss: 0.1954748505105575
Epoch: 5, Loss: 0.1590517331659794
Epoch: 6, Loss: 0.12664253367111086
Epoch: 7, Loss: 0.16918251947810253
Epoch: 8, Loss: 0.13739201599732043
Epoch: 9, Loss: 0.10889729483363529
Epoch: 10, Loss: 0.11478214477499327
Epoch: 11, Loss: 0.0981887572320799
Epoch: 12, Loss: 0.10132899841914575
Epoch: 13, Loss: 0.08251337277547767
Epoch: 14, Loss: 0.09616382846919198
Epoch: 15, Loss: 0.07632221148349344
Epoch: 16, Loss: 0.38806083128166696
Epoch: 17, Loss: 0.2014231001958251
Epoch: 18, Loss: 0.1360605081015577
Epoch: 19, Loss: 0.11924426397308707
Epoch: 20, Loss: 0.10975504327875872
Epoch: 21, Loss: 0.09383280978227655
Epoch: 22, Loss: 0.10707920700466882
Epoch: 23, Loss: 0.09444380018860102
Epoch: 24, Loss: 0.1106104321156939
Epoch: 25, Loss: 0.13825332873811325
Epoch: 26, Loss: 0.08409895966760814
Epoch: 27, Loss: 0.07109163571769993
Epoch: 28, Loss: 0.

In [ ]:
# set model to eval
model.eval()

model(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
# evaluation code
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_label in test_loader:
        outputs = model(batch_features)

        _, predicted = torch.max(outputs, 1)

        total += batch_label.shape[0]

        correct += (predicted == batch_label).sum().item()

print(correct/total)

0.84
